# 基础功能介绍

`npugraph_ex` 是 `torch.compile` 的昇腾图模式后端，通过 NPUGraph 的 Capture-Replay 机制减少 Host 调度开销，并通过 `options` 配置内存复用和 FX 图优化等能力。建议先在最小模型上验证选项是否生效，再迁移到完整模型，并通过精度对比和 Profiling 评估实际收益。

## 1. 常用基础功能

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr><th align="left">Option</th><th align="left">功能描述</th><th align="left">推荐应用场景</th></tr>
  <tr><td align="left"><code>force_eager</code></td><td align="left">保留 <code>npugraph_ex</code> 的上层图增强处理，但不执行 NPUGraph Capture-Replay，改为以 Eager 方式执行 FX Graph。默认值为 <code>False</code>。</td><td align="left">用于区分上层图变换问题与 NPUGraph Runtime 问题。</td></tr>
  <tr><td align="left"><code>force_recapture</code></td><td align="left">设置为 <code>True</code> 时，每次执行前都重新捕获 NPUGraph；默认值为 <code>False</code>，仅在必要时重捕获。</td><td align="left">用于区分图捕获执行问题与输入输出内存处理问题。</td></tr>
  <tr><td align="left"><code>clone_input</code></td><td align="left">针对 user_inputs 类输入进行 clone，并对 clone 后的输入进行内存复用。默认值为 <code>True</code>。</td><td align="left">若 user_inputs 类输入占用内存较大，拷贝后可能导致内存不足。</td></tr>
  <tr><td align="left"><code>clone_output</code></td><td align="left">克隆输出，避免前一次 Replay 的输出内存被后续 Replay 覆盖。默认值为 <code>False</code>；与输入存在别名关系的输出不会被 clone。</td><td align="left">适用于需要跨 Replay 长时间持有输出的场景。</td></tr>
  <tr><td align="left"><code>inplace_pass</code></td><td align="left">将模型中间节点中的非原地算子替换为原地算子。默认值为 <code>True</code>。</td><td align="left">减少计算过程中的内存搬运，从而提升性能。</td></tr>
  <tr><td align="left"><code>input_inplace_pass</code></td><td align="left">将输入相关的“非原地算子 + <code>copy_</code> 算子”恢复为原地算子。默认值为 <code>True</code>。</td><td align="left">减少 KV Cache 等输入更新场景的数据搬运。</td></tr>
  <tr><td align="left"><code>pattern_fusion_pass</code></td><td align="left">通过特定替换规则，使用融合算子替换 FX 图中的多个算子。默认值为 <code>True</code>。</td><td align="left">减少不必要的下发开销，提高模型执行效率。</td></tr>
</table>
<div style="clear: both;"></div>

`fullgraph=True`适合希望整段逻辑 Capture 为整图的模型；若包含暂不支持入图的 Python 逻辑时，可先用`fullgraph=False`定位代码中图断点位置。
`dynamic=False`针对固定 shape 推理场景。

## 2. 使用示例

首次执行包含 Dynamo、Capture 等编译开销，后续执行才适合比较 Replay 性能。

In [ ]:
import torch
import torch_npu  # 注册昇腾 NPU 设备及 npugraph_ex 编译后端

# 在执行示例前确认 NPU 环境可用，并固定随机种子以便复现实验结果。
assert torch.npu.is_available(), "请在已安装 CANN 和 torch_npu 的昇腾环境中运行"
torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("Device:", torch.npu.get_device_name(0))


# 使用简单的 Add + ReLU 模型演示 npugraph_ex 的基础编译流程。
class AddRelu(torch.nn.Module):
    def forward(self, x, y):
        return torch.relu(x + y)


# 模型和输入必须位于同一 NPU 设备。
model = AddRelu().npu()
compiled = torch.compile(
    model,
    backend="npugraph_ex",
    options={
        "inplace_pass": True,        # 优化模型中间节点的原地计算
        "input_inplace_pass": True,  # 恢复受支持的模型输入原地更新
        "pattern_fusion_pass": True, # 启用内置及用户注册的模式融合规则
    },
    fullgraph=True,  # 要求目标逻辑形成完整 FX Graph，图中断时直接报错
    dynamic=False,  # 本示例输入 Shape 固定，使用静态 Shape 编译
)

# 构造固定 Shape 的半精度 NPU 输入。
x = torch.randn(4, 16, dtype=torch.float16).npu()
y = torch.randn(4, 16, dtype=torch.float16).npu()
# 第一次调用触发 FX Graph 编译与 NPUGraph Capture，后续调用进入 Replay。
for step in range(3):
    out = compiled(x, y)
    # NPU 默认异步执行；读取结果前同步，确保本轮计算已经完成。
    torch.npu.synchronize()
    print(step, tuple(out.shape), out.float().mean().item())


## 3. 问题定界

1. 先用 Eager 运行，确认模型本身的数值正确性。
2. 使用 `backend="aot_eager"` 和 `fullgraph=True`，确认用户脚本能够被 PyTorch 编译为整图。
3. 使用 `force_eager=True`，区分 `npugraph_ex` 上层图处理与 NPUGraph Runtime 问题。
4. 使用 `force_recapture=True`，区分图捕获执行与输入输出内存处理问题。
5. 最后逐项调整内存和 FX Pass，并用相同输入做精度校验。

`force_eager` 和 `force_recapture` 是问题定位开关，不应作为生产环境的常驻性能配置。每次只调整一个变量，并保持输入、随机种子和精度阈值一致。

参考：[npugraph_ex 基础功能](https://gitcode.com/Ascend/torchair/tree/master/docs/zh/npugraph_ex/basic)。

## 4. 课后练习

### 一、单选题

（1）【单选题】`force_eager=True` 的主要作用是什么？
- A. 强制每次重新 Capture 图
- B. 保留上层图增强处理，但不执行 NPUGraph Capture-Replay，改为以 Eager 方式执行 FX Graph
- C. 克隆全部模型输出
- D. 为所有图分配共享内存池

（2）【单选题】设置 `force_recapture=True` 后，最符合其行为的是？
- A. 每次 Forward 前重新 Capture NPUGraph
- B. 跳过 Dynamo，直接运行 Python 代码
- C. 仅在首次执行时 Capture
- D. 自动启用所有 FX Pass

（3）【单选题】业务需要在多次 Replay 后仍长期持有某次输出时，应优先开启哪个选项？
- A. clone_input
- B. clone_output
- C. force_eager
- D. force_recapture

（4）【单选题】`fullgraph=True` 的含义是？
- A. 只捕获模型中的第一个算子
- B. 要求将目标函数或模型捕获为单一计算图，遇到图中断时会报错
- C. 自动开启动态 Shape
- D. 仅比较首次编译耗时

（5）【单选题】下列哪个基础选项的默认值为 `False`？
- A. clone_input
- B. clone_output
- C. inplace_pass
- D. pattern_fusion_pass

（6）【单选题】`pattern_fusion_pass=True` 的主要目的是什么？
- A. 强制模型运行在 CPU 上
- B. 使用融合算子替换 FX 图中匹配的多个算子
- C. 在每次 Forward 前重新 Capture
- D. 只对模型输入执行 clone

### 二、多选题

（7）【多选题】进行图模式问题定界时，哪些做法是合理的？
- A. 先以 Eager 结果确认模型数值正确性
- B. 使用 `force_eager=True` 区分上层图处理与 NPUGraph Runtime 问题
- C. 使用 `force_recapture=True` 判断问题是否与回放或地址复用有关
- D. 同时修改全部选项，使问题一次性消失

（8）【多选题】关于 `clone_input=True`，正确的说法有哪些？
- A. 会对 user_inputs 类输入进行 clone
- B. clone 后的输入可用于多张 NPUGraph 间的内存复用
- C. 输入较大时可能带来额外显存压力
- D. 可以消除所有输入拷贝开销

（9）【多选题】比较 Replay 性能时，哪些原则是正确的？
- A. 先建立 Eager 精度基线
- B. 保持输入、随机种子和精度阈值一致
- C. 将首次的 Dynamo/Capture 编译开销与后续 Replay 分开统计
- D. 只要 FX 图节点减少，就不必进行精度校验

（10）【多选题】关于基础 Options 的对应关系，正确的有哪些？
- A. `input_inplace_pass` 面向模型输入相关的原地更新优化
- B. `inplace_pass` 主要处理模型中间节点的原地化机会
- C. `pattern_fusion_pass` 可对匹配的子图执行融合替换
- D. `clone_output` 的目的只是降低首次编译耗时

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/03.02_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
